# RGB–thermal image preparation

Prepare paired RGB and thermal images for the `head_counting` gold dataset. The two files represent the same scene as different layers.

This notebook only inspects, transforms, compares, and exports images.

## 1. Select one RGB/thermal pair

In [ ]:
from pathlib import Path
import sys
import cv2
import matplotlib.pyplot as plt

# Resolve paths from the repository, regardless of the notebook working directory.
NOTEBOOK_DIR = Path.cwd()
REPO_DIR = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / 'app' / 'head_counting').exists()), NOTEBOOK_DIR)
sys.path.insert(0, str(REPO_DIR / 'app'))
from head_counting.preprocessing import RGBTImageEqualizer

# Change these paths to the matching source files.
RGB_PATH = REPO_DIR / 'data' / 'bronze' / 'rgb' / 'image.jpg'
THERMAL_PATH = REPO_DIR / 'data' / 'bronze' / 'thermal' / 'image.jpg'
OUTPUT_DIR = REPO_DIR / 'data' / 'silver' / 'rgbt'

assert RGB_PATH.exists(), f'RGB image not found: {RGB_PATH}'
assert THERMAL_PATH.exists(), f'Thermal image not found: {THERMAL_PATH}'
rgb = cv2.imread(str(RGB_PATH), cv2.IMREAD_COLOR)
thermal = cv2.imread(str(THERMAL_PATH), cv2.IMREAD_COLOR)
assert rgb is not None, f'Could not read RGB image: {RGB_PATH}'
assert thermal is not None, f'Could not read thermal image: {THERMAL_PATH}'
print(f'RGB:     {RGB_PATH} -> {rgb.shape[1]} x {rgb.shape[0]}')
print(f'Thermal: {THERMAL_PATH} -> {thermal.shape[1]} x {thermal.shape[0]}')

## 2. Configure and transform the pair

`mode='homography'` performs feature-based alignment after the RGB field-of-view crop. Use `mode='crop'` when the calibrated center crop is preferred and feature matching is unreliable.

In [ ]:
TARGET_SIZE = (1280, 1024)  # (width, height)
MODE = 'homography'         # 'homography' or 'crop'
FOV_CROP_RATIO = 0.70
RGB_SHIFT_X = -22
RGB_SHIFT_Y = -23

equalizer = RGBTImageEqualizer(target_size=TARGET_SIZE, mode=MODE, fov_crop_ratio=FOV_CROP_RATIO, shift_rgb_x=RGB_SHIFT_X, shift_rgb_y=RGB_SHIFT_Y, thermal_clahe=True, keep_aspect_ratio=True)
rgb_equalized, thermal_equalized = equalizer.process_pair(rgb, thermal)
overlap = equalizer.create_blend_overlay(rgb_equalized, thermal_equalized, alpha=0.5)
print(f'Transformed RGB:     {rgb_equalized.shape[1]} x {rgb_equalized.shape[0]}')
print(f'Transformed thermal: {thermal_equalized.shape[1]} x {thermal_equalized.shape[0]}')

## 3. Inspect the layers and overlap

In [ ]:
def show_bgr(image, title):
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for axis, image, title in zip(axes, [rgb_equalized, thermal_equalized, overlap], ['RGB transformed', 'Thermal transformed', '50/50 overlap']):
    plt.sca(axis)
    show_bgr(image, title)
plt.tight_layout()
plt.show()

## 4. Save the transformed pair

Review the overlap before promoting this pair to the gold dataset.

In [ ]:
output_images = OUTPUT_DIR / 'images'
output_checks = OUTPUT_DIR / 'layer_blend_checks'
output_images.mkdir(parents=True, exist_ok=True)
output_checks.mkdir(parents=True, exist_ok=True)

rgb_output = output_images / f'{RGB_PATH.stem}_equalized{RGB_PATH.suffix}'
thermal_output = output_images / f'{THERMAL_PATH.stem}_equalized.jpg'
overlap_output = output_checks / f'{RGB_PATH.stem}_blend_check.jpg'
assert cv2.imwrite(str(rgb_output), rgb_equalized)
assert cv2.imwrite(str(thermal_output), thermal_equalized)
assert cv2.imwrite(str(overlap_output), overlap)
print(f'RGB:     {rgb_output}')
print(f'Thermal: {thermal_output}')
print(f'Overlap: {overlap_output}')